# 04 — Cross-Trial ValidationThis notebook trains on one experimental trial and tests on the otherto evaluate how well models generalize across independent experiments.- **Approach A**: Train on Trial 1, test on Trial 2- **Approach B**: Train on Trial 2, test on Trial 1This provides a more realistic assessment than random train/test splits,since the trials represent genuinely independent measurements.

In [ ]:
import sys, ossys.path.insert(0, os.path.abspath('..'))import pandas as pdimport numpy as npimport matplotlib.pyplot as pltimport seaborn as snsfrom sklearn.ensemble import RandomForestRegressorfrom sklearn.model_selection import GridSearchCVfrom src.data_loading import load_trial_data, compute_mean_absorbancefrom src.feature_extraction import extract_features_dataframefrom src.models import evaluate_regression, tune_random_forestfrom src.visualization import plot_actual_vs_predicted, plot_feature_importance%matplotlib inline

## 1. Load Trials and Extract Features

In [ ]:
DATA_DIR = os.path.join('..', 'data')df1, df2 = load_trial_data(    os.path.join(DATA_DIR, 'peptide_csv1_last20.csv'),    os.path.join(DATA_DIR, 'peptide2_csv1_last20.csv'),)# Compute mean absorbance per trialy_trial1 = compute_mean_absorbance(df1)y_trial2 = compute_mean_absorbance(df2)# Extract features (same sequences in both trials)sequences = df1['Sequence'].tolist()X = extract_features_dataframe(sequences)print(f'Features shape: {X.shape}')print(f'Trial 1 absorbance range: [{y_trial1.min():.3f}, {y_trial1.max():.3f}]')print(f'Trial 2 absorbance range: [{y_trial2.min():.3f}, {y_trial2.max():.3f}]')

## 2. Approach A — Train on Trial 1, Test on Trial 2

In [ ]:
# Train RF on Trial 1rf_a = RandomForestRegressor(random_state=42)rf_a.fit(X, y_trial1)y_pred_a = rf_a.predict(X)metrics_a = evaluate_regression(y_trial2.values, y_pred_a)print('Trial 1 → Trial 2:')print(f"  R²:   {metrics_a['r2']:.4f}")print(f"  RMSE: {metrics_a['rmse']:.4f}")print(f"  MAE:  {metrics_a['mae']:.4f}")

In [ ]:
# Tune with GridSearchCVparam_grid = {    'n_estimators': [50, 100, 200],    'max_depth': [None, 10, 20],    'min_samples_split': [2, 5],    'min_samples_leaf': [1, 2],}grid_a = GridSearchCV(    RandomForestRegressor(random_state=42),    param_grid, cv=5, scoring='r2', n_jobs=-1, verbose=1,)grid_a.fit(X, y_trial1)print(f'Best params: {grid_a.best_params_}')print(f'Best CV R²: {grid_a.best_score_:.4f}')y_pred_a_tuned = grid_a.predict(X)metrics_a_tuned = evaluate_regression(y_trial2.values, y_pred_a_tuned)print(f"\nTuned — R²: {metrics_a_tuned['r2']:.4f}, RMSE: {metrics_a_tuned['rmse']:.4f}, MAE: {metrics_a_tuned['mae']:.4f}")

In [ ]:
fig = plot_actual_vs_predicted(    y_trial2.values, y_pred_a_tuned,    title='Trial 1 → Trial 2: Actual vs Predicted',)plt.show()

## 3. Approach B — Train on Trial 2, Test on Trial 1

In [ ]:
grid_b = GridSearchCV(    RandomForestRegressor(random_state=42),    param_grid, cv=5, scoring='r2', n_jobs=-1, verbose=1,)grid_b.fit(X, y_trial2)print(f'Best params: {grid_b.best_params_}')print(f'Best CV R²: {grid_b.best_score_:.4f}')y_pred_b_tuned = grid_b.predict(X)metrics_b_tuned = evaluate_regression(y_trial1.values, y_pred_b_tuned)print(f"\nTuned — R²: {metrics_b_tuned['r2']:.4f}, RMSE: {metrics_b_tuned['rmse']:.4f}, MAE: {metrics_b_tuned['mae']:.4f}")

In [ ]:
fig = plot_actual_vs_predicted(    y_trial1.values, y_pred_b_tuned,    title='Trial 2 → Trial 1: Actual vs Predicted',)plt.show()

## 4. Cross-Trial Comparison

In [ ]:
comparison = pd.DataFrame({    'Sequence': sequences,    'True_Trial1': y_trial1.values,    'Pred_Trial1': y_pred_b_tuned,    'Error_Trial1': np.abs(y_trial1.values - y_pred_b_tuned),    'True_Trial2': y_trial2.values,    'Pred_Trial2': y_pred_a_tuned,    'Error_Trial2': np.abs(y_trial2.values - y_pred_a_tuned),})comparison = comparison.sort_values('Error_Trial2', ascending=False)print('Sequences with highest prediction error (Trial 1 → Trial 2):')comparison.head(10)

## 5. Feature Importance

In [ ]:
if hasattr(grid_a.best_estimator_, 'feature_importances_'):    fig = plot_feature_importance(        grid_a.best_estimator_.feature_importances_,        list(X.columns),        n_top=15,    )    plt.title('Feature Importances (Trial 1 → Trial 2 Model)')    plt.show()

## 6. Predict New Sequences

In [ ]:
new_seq = 'ENEKNQDSEAINRRA'feats = extract_features_dataframe([new_seq])prediction = grid_a.predict(feats)print(f'Predicted absorbance for {new_seq}: {prediction[0]:.4f}')